In [1]:
# import libraries
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dotenv

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Pushover notifications app
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [4]:
# Setup pushover notifications
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
# Record user details
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interests from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [6]:
#  Record unknown questions
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [7]:
# Record user details tool
record_user_details_json = {
    "name": "record_user_details",
    "description": "Call this tool when a user provides their email or asks to be contacted for follow-up.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "format": "email",
                "description": "The user's valid email address"
            },
            "name": {
                "type": "string",
                "description": "The user's name if explicitly provided"
            },
            "notes": {
                "type": "string",
                "description": "Relevant context or summary of the conversation"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [8]:
# Record unknown questions tool
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Call this tool when a user asks a question that you cannot confidently answer or lack sufficient information to respond accurately.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The exact user question that could not be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [9]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [10]:
# Function to take a list of tool calls and run them

import json

TOOL_REGISTRY = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unknown_question, 
}

def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            result = {"error": "Invalid JSON arguments"}
            arguments = {}

        print(f"Tool called: {tool_name}", flush=True)

        tool_function = TOOL_REGISTRY.get(tool_name)

        if not tool_function:
            result = {"error": f"Unknown tool: {tool_name}"}
        else:
            try:
                result = tool_function(**arguments)
            except Exception as e:
                result = {
                    "error": str(e),
                    "tool": tool_name
                }

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return results

In [11]:
# Open and Read contents of CV in PDF file format
reader = PdfReader("data/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("data/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Alexander Karari."

In [12]:
# System prompt
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s CV, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the CV as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the CV. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # LLM call
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # Let LLM call a tool
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [14]:
# Implement gradio
# gr.ChatInterface(chat, type="messages").launch()

# --- Chat Wrapper ---
def respond(message, history):
    # Convert Gradio history (tuples) → OpenAI format
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # Call your existing chat function
    response = chat(message, messages)

    # Append in Gradio tuple format
    history.append((message, response))

    return "", history

with gr.Blocks(theme=gr.themes.Soft(), title="AI CV Agent") as demo:
    # 🔷 HERO SECTION
    gr.Markdown(""" 
    # 🤖 AI CV Agent
    ### Turn a static resume into a conversation

    Ask anything about my experience, skills, or projects.
    This assistant represents my professional profile using AI.

    ---
    """)

    # 🔘 CTA buttons (guidance)
    with gr.Row():
        btn1 = gr.Button("💼 What experience do you have?")
        btn2 = gr.Button("🧠 What are your key skills?")
        btn3 = gr.Button("🚀 What projects have you worked on?")

    # 💬 MAIN CHAT AREA
    chatbot = gr.Chatbot(height=450, bubble_full_width=False)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask me anythong about my background...",
            container=False,
            scale=8
        )
        send = gr.Button("Send", variant="primary", scale=1)

    # 🔁 Bind chat
    send.click(respond, [msg, chatbot], [msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

     # 🔘 Button interactions (pre-fill questions)
    btn1.click(lambda: "What experience do you have?", None, msg)
    btn2.click(lambda: "What are your key skills?", None, msg)
    btn3.click(lambda: "What projects have you worked on?", None, msg)

    # 🧠 HOW IT WORKS
    gr.Markdown("""
    ---
    ## 🧠 How it works

    This is not just a chatbot.

    It’s a lightweight **AI agent** that:
    - Understands your question  
    - Decides whether to respond or take action  
    - Can capture contact details or log unanswered questions  

    Built to demonstrate real-world applications of LLMs beyond text generation.
    """)

    # 📊 ABOUT / CONTEXT SECTION
    gr.Markdown(f"""
    ---
    ## 👤 About Me

    {summary[:500]}...
    """)

    # 📩 CONTACT CTA
    gr.Markdown("""
    ---
    ## 📬 Get in touch

    If you're interested in working together or want to follow up,  
    feel free to share your email in the chat.

    The assistant will capture it and I’ll reach out.
    """)

    # 🔻 FOOTER
    gr.Markdown("""
    ---
    ⚡ Built with LLMs, tool-calling, and Gradio  
    🚀 Designed as an interactive AI agent, not a static resume  
    """)

demo.launch()




    



/var/folders/bs/dbv1bzdd4qq0fhjqbb4z44hh0000gn/T/ipykernel_28007/1808598530.py:39: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450, bubble_full_width=False)
/var/folders/bs/dbv1bzdd4qq0fhjqbb4z44hh0000gn/T/ipykernel_28007/1808598530.py:39: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(height=450, bubble_full_width=False)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
